## 與 code2 的差異

- 參考：https://docs.google.com/document/d/1gyX_B6hePucgCHeOlSFD5GQi5XfMGTPAZVR7J0zWP7w/edit?tab=t.lmys5o9oj10c#heading=h.vqzcjps2llcx
- 移除極端離群值
- 擴充高級特徵 與 修正數值型偏態
- 引入新樹模型 (LightGBM / CatBoost)
- 調整最後的模型加權比例


## 1. setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sklearn 工具與前處理套件
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# 模型
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

print("Setup Complete")

## 2. 資料讀取

- 房價資料集中，許多類別欄位的缺失值（NaN）代表本就沒有該設施（如無車庫、無地下室、無游泳池），若直接補眾數會破壞實際物理意義，因此做獨立標記。
- PoolQC、GarageType 等欄位的 NaN 轉為 'None'，不會被 SimpleImputer 錯誤填補為常見設施。
- 缺失值處理 (刪除法)：只針對 Target 變數 X_full.dropna(subset=['SalePrice']) 刪除目標值缺失的列。


In [ ]:
# 本地
# base_path = (
#     r"C:\Users\User\Documents\GitHub\template-kaggle-competition\housing-prices\data"
# )
# X_full = pd.read_csv(f"{base_path}\\train.csv", index_col="Id")
# X_test_full = pd.read_csv(f"{base_path}\\test.csv", index_col="Id")

# kaggle
X_full = pd.read_csv(
    "/kaggle/input/competitions/home-data-for-ml-course/train.csv", index_col="Id"
)
X_test_full = pd.read_csv(
    "/kaggle/input/competitions/home-data-for-ml-course/test.csv", index_col="Id"
)

# 移除極端離群值
outliers = X_full[(X_full["GrLivArea"] > 4000) & (X_full["SalePrice"] < 300000)].index
X_full.drop(outliers, axis=0, inplace=True)

# 移除沒有目標值的資料
X_full.dropna(axis=0, subset=["SalePrice"], inplace=True)

# 目標變數取 Log 對數轉換
y_log = np.log1p(X_full.SalePrice)
X_full.drop(["SalePrice"], axis=1, inplace=True)

# 針對具備實際物理意義缺失值的欄位補上 'None' 或 0
none_cols = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
]

for col in none_cols:
    if col in X_full.columns:
        X_full[col] = X_full[col].fillna("None")
        X_test_full[col] = X_test_full[col].fillna("None")

zero_cols = [
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
]
for col in zero_cols:
    if col in X_full.columns:
        X_full[col] = X_full[col].fillna(0)
        X_test_full[col] = X_test_full[col].fillna(0)

# 顯示資料筆數與欄位數
print(f"訓練集形狀: {X_full.shape}")
print(f"測試集形狀: {X_test_full.shape}")

""" >>>
訓練集形狀: (1458, 79)
測試集形狀: (1459, 79)
"""

## 3. 特徵工程

- 組合出反應總面積、總衛浴數與房屋折舊狀況的高影響力特徵。


In [ ]:
def add_custom_features(df):
    df_out = df.copy()
    if "MSSubClass" in df_out.columns:
        df_out["MSSubClass"] = df_out["MSSubClass"].astype(str)

    # 空間與面積
    df_out["TotalSF"] = df_out["TotalBsmtSF"] + df_out["1stFlrSF"] + df_out["2ndFlrSF"]
    df_out["TotalBaths"] = (
        df_out["FullBath"]
        + (0.5 * df_out["HalfBath"])
        + df_out["BsmtFullBath"]
        + (0.5 * df_out["BsmtHalfBath"])
    )
    df_out["TotalOutsideSF"] = (
        df_out["WoodDeckSF"]
        + df_out["OpenPorchSF"]
        + df_out["EnclosedPorch"]
        + df_out["3SsnPorch"]
        + df_out["ScreenPorch"]
    )

    # 時間與屋齡
    df_out["HouseAge"] = df_out["YrSold"] - df_out["YearBuilt"]
    df_out["RemodAge"] = df_out["YrSold"] - df_out["YearRemodAdd"]
    df_out["IsRemodeled"] = (df_out["YearBuilt"] != df_out["YearRemodAdd"]).astype(int)

    # 比例與狀態
    df_out["Pct_LivArea_Total"] = df_out["GrLivArea"] / (df_out["TotalSF"] + 1e-5)
    df_out["HasBasement"] = (df_out["TotalBsmtSF"] > 0).astype(int)
    df_out["HasGarage"] = (df_out["GarageArea"] > 0).astype(int)

    # 互動特徵
    df_out["OverallQual_GrLivArea"] = df_out["OverallQual"] * df_out["GrLivArea"]
    df_out["OverallQual_TotalSF"] = df_out["OverallQual"] * df_out["TotalSF"]
    df_out["GarageScore"] = df_out["GarageCars"] * df_out["GarageArea"]
    df_out["OverallCond_HouseAge"] = df_out["OverallCond"] * df_out["HouseAge"]

    return df_out


X_full_fe = add_custom_features(X_full)
X_test_full_fe = add_custom_features(X_test_full)

## 4. 特徵分群 (數值型與類別型) 與 編碼定義


In [ ]:
ordinal_cols = [
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
]
qual_order = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
ordinal_categories = [qual_order for _ in ordinal_cols]

categorical_cols = [
    cname
    for cname in X_full_fe.columns
    if X_full_fe[cname].dtype == "object" and cname not in ordinal_cols
]
numerical_cols = [
    cname
    for cname in X_full_fe.columns
    if X_full_fe[cname].dtype in ["int64", "float64"]
]

my_cols = numerical_cols + ordinal_cols + categorical_cols
X = X_full_fe[my_cols].copy()
X_test = X_test_full_fe.reindex(columns=X.columns)

""" >>>
選取的數值特徵數: 44
選取的類別特徵數: 35
訓練集與測試集順序是否完全一致: True
"""

## 5. 資料前處理管道

- 使用 ColumnTransformer 分別處理數值、有序類別與無序類別。
- 將品質欄位定義明確的等級關係，使用 OrdinalEncoder；無順序性的普通文字欄位使用 OneHotEncoder。
- 缺失值處理 (插補法)：數值型用 SimpleImputer(strategy='median')；類別型用 strategy='most_frequent'。
- One-Hot Encoding (獨熱編碼)：對低基數（< 10 個獨特值）的類別欄位套用 OneHotEncoder。


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import skew


# 宣告防止 Data Leakage 的偏態轉換器
class SkewnessTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.75):
        self.threshold = threshold
        self.skewed_features_ = []

    def fit(self, X, y=None):
        exclude_feats = [
            "OverallQual_GrLivArea",
            "OverallQual_TotalSF",
            "GarageScore",
            "OverallCond_HouseAge",
            "IsRemodeled",
            "HasBasement",
            "HasGarage",
        ]
        self.skewed_features_ = []
        for col in X.columns:
            if col not in exclude_feats and X[col].dtype in ["int64", "float64"]:
                if abs(skew(X[col].dropna())) > self.threshold:
                    self.skewed_features_.append(col)
        return self

    def transform(self, X):
        X_out = X.copy()
        for col in self.skewed_features_:
            if col in X_out.columns:
                X_out[col] = np.log1p(np.maximum(0, X_out[col]))
        return X_out


# --- 1. 樹模型專用管道 (獨立宣告，絕對不與線性模型共用變數) ---
# 完全獨立宣告，拒絕共用變數
ord_transformer_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)
cat_transformer_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
num_transformer_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("skew_corr", SkewnessTransformer(threshold=0.75)),
    ]
)
preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", num_transformer_tree, numerical_cols),
        ("ord", ord_transformer_tree, ordinal_cols),
        ("cat", cat_transformer_tree, categorical_cols),
    ]
)

# --- 2. 線性模型專用組件 (全新宣告，與樹模型完全隔開) ---
ord_transformer_linear = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)
cat_transformer_linear = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
num_transformer_linear = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("skew_corr", SkewnessTransformer(threshold=0.75)),
        ("scaler", StandardScaler()),  # 線性專用縮放
    ]
)
preprocessor_linear = ColumnTransformer(
    transformers=[
        ("num", num_transformer_linear, numerical_cols),
        ("ord", ord_transformer_linear, ordinal_cols),
        ("cat", cat_transformer_linear, categorical_cols),
    ]
)

## 6. 全局驗證設定與 RMSLE 評估工具

In [ ]:
from sklearn.metrics import root_mean_squared_log_error


class Config:
    N_SPLITS = 10
    RANDOM_STATE = 42
    SHUFFLE = True
    XGB_PARAMS = {
        "n_estimators": 1500,
        "learning_rate": 0.05,
        "max_depth": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "n_jobs": -1,
    }

    # LightGBM 超參數
    LGB_PARAMS = {
        "n_estimators": 1500,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    # CatBoost 超參數
    CAT_PARAMS = {
        "iterations": 1500,
        "learning_rate": 0.05,
        "depth": 6,
        "random_state": 42,
        "verbose": False,
    }


def evaluate_rmsle(y_true_log, y_pred_log):
    mse = mean_squared_error(y_true_log, y_pred_log)
    return np.sqrt(mse)

## 7. 建立線性模型群 (Lasso / ElasticNet / Ridge) OOF 函式

In [ ]:
from sklearn.linear_model import Lasso, ElasticNet, Ridge
from sklearn.model_selection import KFold


def run_linear_oof_training(X, y_log, X_test, preprocessor):
    models = {
        "Lasso": Lasso(alpha=0.0005, max_iter=10000, random_state=42),
        "ElasticNet": ElasticNet(
            alpha=0.0005, l1_ratio=0.5, max_iter=10000, random_state=42
        ),
        "Ridge": Ridge(alpha=12),
    }
    oof_preds = {name: np.zeros(len(X)) for name in models}
    test_preds = {name: np.zeros(len(X_test)) for name in models}
    kf = KFold(
        n_splits=Config.N_SPLITS,
        shuffle=Config.SHUFFLE,
        random_state=Config.RANDOM_STATE,
    )

    print(f"開始線性模型群 {Config.N_SPLITS}-Fold OOF 交叉驗證")
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
        X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y_log.iloc[val_idx]

        X_train_trans = preprocessor.fit_transform(X_train)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test)

        for name, model in models.items():
            model.fit(X_train_trans, y_train)
            oof_preds[name][val_idx] = model.predict(X_val_trans)
            test_preds[name] += model.predict(X_test_trans) / Config.N_SPLITS

    for name in models:
        score = evaluate_rmsle(y_log, oof_preds[name])
        print(f"{name} Overall OOF RMSLE: {score:.5f}")
    return oof_preds, test_preds

## 8. 建立 XGBoost 模型 OOF 函式

In [ ]:
# 建立 3 大樹模型 OOF 訓練函式
from sklearn.model_selection import KFold
import numpy as np
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, early_stopping
from catboost import CatBoostRegressor


# --- 8A. XGBoost 訓練函式 ---
def run_xgb_oof_training(X, y_log, X_test, preprocessor):
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    kf = KFold(
        n_splits=Config.N_SPLITS,
        shuffle=Config.SHUFFLE,
        random_state=Config.RANDOM_STATE,
    )
    print(f"開始 XGBoost 模型 {Config.N_SPLITS}-Fold OOF 交叉驗證")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
        X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y_log.iloc[val_idx]

        # 樹模型前處理
        X_train_trans = preprocessor.fit_transform(X_train)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test)

        model = XGBRegressor(**Config.XGB_PARAMS)
        model.fit(
            X_train_trans,
            y_train,
            eval_set=[(X_val_trans, y_val)],
            early_stopping_rounds=100,  # 100代無改善即停止
            verbose=False,
        )

        oof_preds[val_idx] = model.predict(X_val_trans)
        test_preds += model.predict(X_test_trans) / Config.N_SPLITS

    print(f"XGBoost Overall OOF RMSLE: {evaluate_rmsle(y_log, oof_preds):.5f}\n")
    return oof_preds, test_preds


# --- 8B. LightGBM 訓練函式 ---
def run_lgb_oof_training(X, y_log, X_test, preprocessor):
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    kf = KFold(
        n_splits=Config.N_SPLITS,
        shuffle=Config.SHUFFLE,
        random_state=Config.RANDOM_STATE,
    )
    print(f"開始 LightGBM 模型 {Config.N_SPLITS}-Fold OOF 交叉驗證")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
        X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y_log.iloc[val_idx]

        # 樹模型前處理
        X_train_trans = preprocessor.fit_transform(X_train)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test)

        model = LGBMRegressor(**Config.LGB_PARAMS)
        model.fit(
            X_train_trans,
            y_train,
            eval_set=[(X_val_trans, y_val)],
            callbacks=[
                early_stopping(stopping_rounds=100, verbose=False)
            ],  # LightGBM 專用早停寫法
        )

        oof_preds[val_idx] = model.predict(X_val_trans)
        test_preds += model.predict(X_test_trans) / Config.N_SPLITS

    print(f"LightGBM Overall OOF RMSLE: {evaluate_rmsle(y_log, oof_preds):.5f}\n")
    return oof_preds, test_preds


# --- 8C. CatBoost 訓練函式 ---
def run_cat_oof_training(X, y_log, X_test, preprocessor):
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    kf = KFold(
        n_splits=Config.N_SPLITS,
        shuffle=Config.SHUFFLE,
        random_state=Config.RANDOM_STATE,
    )
    print(f"開始 CatBoost 模型 {Config.N_SPLITS}-Fold OOF 交叉驗證")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
        X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y_log.iloc[val_idx]

        # 樹模型前處理
        X_train_trans = preprocessor.fit_transform(X_train)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test)

        model = CatBoostRegressor(**Config.CAT_PARAMS)
        model.fit(
            X_train_trans,
            y_train,
            eval_set=[(X_val_trans, y_val)],
            early_stopping_rounds=100,  # CatBoost 早停機制
            verbose=False,
        )

        oof_preds[val_idx] = model.predict(X_val_trans)
        test_preds += model.predict(X_test_trans) / Config.N_SPLITS

    print(f"CatBoost Overall OOF RMSLE: {evaluate_rmsle(y_log, oof_preds):.5f}\n")
    return oof_preds, test_preds

## 9. 執行訓練與模型融合

In [ ]:
from scipy.optimize import minimize

# 彙整 OOF 矩陣
oof_matrix = np.column_stack(
    [oof_xgb, oof_lgb, oof_cat, oof_linears["Lasso"], oof_linears["Ridge"]]
)

# 彙整測試集預測矩陣
test_matrix = np.column_stack(
    [
        test_preds_xgb,
        test_preds_lgb,
        test_preds_cat,
        test_preds_linears["Lasso"],
        test_preds_linears["Ridge"],
    ]
)

model_names = ["XGBoost", "LightGBM", "CatBoost", "Lasso", "Ridge"]


# 優化目標函數
def loss_function(weights):
    blend_oof_preds = np.dot(oof_matrix, weights)
    return evaluate_rmsle(y_log, blend_oof_preds)


# 限制條件
constraints = {"type": "eq", "fun": lambda w: 1.0 - np.sum(w)}
bounds = [(0, 1)] * len(model_names)
initial_weights = [1.0 / len(model_names)] * len(model_names)

# 求解
optimization_result = minimize(
    loss_function,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)
best_weights = optimization_result.x

print("\n 權重優化完成")
for name, weight in zip(model_names, best_weights):
    print(f"{name} 最佳融合權重: {weight:.4f}")
print(f"融合模型最終（優化後）OOF RMSLE: {optimization_result.fun:.5f}\n")

## 10. 預測並匯出

In [ ]:
# 生成最終測試集預測值
final_test_preds_log = np.dot(test_matrix, best_weights)
final_preds_dollar = np.expm1(final_test_preds_log)

# 檢查與匯出
print("測試集有無空值：", np.isnan(final_preds_dollar).any())
print("預測值描述統計：\n", pd.Series(final_preds_dollar).describe())

""" >>>
測試集有無空值： False
預測值描述統計：
 count    1.459000e+03
mean     2.308868e+05
std      1.177833e+05
min      5.316360e+04
25%      1.572835e+05
50%      1.973764e+05
75%      2.686407e+05
max      1.124539e+06
dtype: float64
"""

In [ ]:
output = pd.DataFrame({"Id": X_test.index, "SalePrice": final_preds_dollar})
output.to_csv("submission.csv", index=False)
print("\nsubmission.csv 匯出成功！")
print(output.head())

""" >>>
submission.csv 匯出成功！格式如下：
     Id      SalePrice
0  1461  171991.834255
1  1462  229125.779210
2  1463  244892.123496
3  1464  265142.498988
4  1465  220844.224195
"""